## Áp dụng Quantization với TensorFlow Lite

In [ ]:
# Chuyển đổi mô hình sang dạng INT8 để tăng tốc độ xử lý và giảm kích thước mô hình

import tensorflow as tf
from tensorflow import lite
from tensorflow.lite.python.interpreter import Interpreter
import numpy as np
import time
import os

In [ ]:
print("🔧 Bắt đầu quá trình Quantization...")

In [ ]:
# 1. Load mô hình đã train
print("📥 Loading mô hình đã train...")
model = tf.keras.models.load_model('attack_classifier.h5', custom_objects={'AttentionLayer': AttentionLayer})


In [ ]:
# 2. Tạo representative dataset cho quantization
print("📊 Tạo representative dataset cho quantization...")
def representative_dataset():
    # Sử dụng một phần dữ liệu test để làm representative dataset
    num_calibration_samples = 1000
    calibration_data = X_attack_test_scaled_no[:num_calibration_samples]
    
    for data in calibration_data:
        # Reshape data để phù hợp với input shape của model
        yield [data.reshape(1, -1, 1)]

In [ ]:
# 3. Chuyển đổi sang TensorFlow Lite với Quantization INT8
print("⚡ Chuyển đổi sang TensorFlow Lite với Quantization INT8...")
converter = lite.TFLiteConverter.from_keras_model(model)

# Cấu hình quantization
converter.optimizations = [lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.target_spec.supported_types = [tf.int8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

In [ ]:
# Chuyển đổi model
tflite_model_quantized = converter.convert()

In [ ]:
# 4. Lưu model đã quantized
print("💾 Lưu model đã quantized...")
with open('attack_classifier_quantized.tflite', 'wb') as f:
    f.write(tflite_model_quantized)


In [ ]:
# 5. So sánh kích thước model
original_size = os.path.getsize('attack_classifier.h5')
quantized_size = os.path.getsize('attack_classifier_quantized.tflite')

print(f"\n📏 So sánh kích thước model:")
print(f"   Original model (H5): {original_size / (1024*1024):.2f} MB")
print(f"   Quantized model (TFLite): {quantized_size / (1024*1024):.2f} MB")
print(f"   Giảm kích thước: {((original_size - quantized_size) / original_size * 100):.1f}%")

In [ ]:
# 6. Test inference với model đã quantized
print("\n🧪 Test inference với model đã quantized...")

In [ ]:
# Load TFLite model
interpreter = Interpreter(model_path='attack_classifier_quantized.tflite')
interpreter.allocate_tensors()


In [ ]:
# Lấy input và output details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(f"Input details: {input_details}")
print(f"Output details: {output_details}")


In [ ]:
# Test với một số mẫu
test_samples = X_attack_test_scaled_no[:10]
original_predictions = []
quantized_predictions = []

print("\n📈 So sánh độ chính xác giữa model gốc và model đã quantized:")

In [ ]:
# Test model gốc
start_time = time.time()
original_pred = model.predict(test_samples)
original_time = time.time() - start_time


In [ ]:
# Test model đã quantized
start_time = time.time()
for sample in test_samples:
    # Reshape và quantize input
    input_data = sample.reshape(1, -1, 1)
    
    # Quantize input data
    input_scale, input_zero_point = input_details[0]['quantization']
    input_data_quantized = np.round(input_data / input_scale + input_zero_point).astype(np.int8)
    
    # Set input tensor
    interpreter.set_tensor(input_details[0]['index'], input_data_quantized)
    
    # Run inference
    interpreter.invoke()
    
    # Get output
    output_data = interpreter.get_tensor(output_details[0]['index'])
    
    # Dequantize output
    output_scale, output_zero_point = output_details[0]['quantization']
    output_data_dequantized = (output_data.astype(np.float32) - output_zero_point) * output_scale
    
    quantized_predictions.append(output_data_dequantized)

quantized_time = time.time() - start_time

In [ ]:
# So sánh kết quả
print(f"\n⏱️  Thời gian inference:")
print(f"   Original model: {original_time:.4f} seconds")
print(f"   Quantized model: {quantized_time:.4f} seconds")
print(f"   Tốc độ tăng: {original_time/quantized_time:.2f}x")

In [ ]:
# So sánh độ chính xác
original_pred_classes = np.argmax(original_pred, axis=1)
quantized_pred_classes = np.argmax(np.array(quantized_predictions).squeeze(), axis=1)

accuracy_comparison = np.mean(original_pred_classes == quantized_pred_classes)
print(f"\n🎯 Độ chính xác so sánh: {accuracy_comparison:.4f}")

In [ ]:
# 7. Lưu thông tin quantization
quantization_info = {
    'original_size_mb': original_size / (1024*1024),
    'quantized_size_mb': quantized_size / (1024*1024),
    'size_reduction_percent': ((original_size - quantized_size) / original_size * 100),
    'speedup_factor': original_time/quantized_time,
    'accuracy_comparison': accuracy_comparison,
    'input_quantization': {
        'scale': input_details[0]['quantization'][0],
        'zero_point': input_details[0]['quantization'][1]
    },
    'output_quantization': {
        'scale': output_details[0]['quantization'][0],
        'zero_point': output_details[0]['quantization'][1]
    }
}

import json
with open('quantization_info.json', 'w') as f:
    json.dump(quantization_info, f, indent=2)

print(f"\n✅ Hoàn thành Quantization!")
print(f"📁 Files đã tạo:")
print(f"   - attack_classifier_quantized.tflite (model đã quantized)")
print(f"   - quantization_info.json (thông tin chi tiết)")

In [ ]:
# 8. Tạo function để sử dụng model đã quantized
def load_quantized_model(model_path='attack_classifier_quantized.tflite'):
    """
    Load và trả về model đã quantized
    """
    interpreter = Interpreter(model_path=model_path)
    interpreter.allocate_tensors()
    return interpreter

def predict_with_quantized_model(interpreter, input_data, scaler, label_encoder):
    """
    Dự đoán sử dụng model đã quantized
    """
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    
    # Preprocess input data
    input_scaled = scaler.transform(input_data)
    input_reshaped = input_scaled.reshape(-1, input_scaled.shape[1], 1)
    
    predictions = []
    
    for sample in input_reshaped:
        # Quantize input
        input_scale, input_zero_point = input_details[0]['quantization']
        input_quantized = np.round(sample / input_scale + input_zero_point).astype(np.int8)
        
        # Set input tensor
        interpreter.set_tensor(input_details[0]['index'], input_quantized.reshape(1, -1, 1))
        
        # Run inference
        interpreter.invoke()
        
        # Get output
        output_data = interpreter.get_tensor(output_details[0]['index'])
        
        # Dequantize output
        output_scale, output_zero_point = output_details[0]['quantization']
        output_dequantized = (output_data.astype(np.float32) - output_zero_point) * output_scale
        
        predictions.append(output_dequantized)
    
    predictions = np.array(predictions).squeeze()
    predicted_classes = np.argmax(predictions, axis=1)
    predicted_labels = label_encoder.inverse_transform(predicted_classes)
    
    return predicted_labels, predictions

print(f"\n🔧 Function để sử dụng model đã quantized đã được tạo!")
print(f"   - load_quantized_model(): Load model đã quantized")
print(f"   - predict_with_quantized_model(): Dự đoán với model đã quantized")
